In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
X = pd.read_csv("train.csv")
y = X.pop("label")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [3]:
class Layer:
    def __call__(self, prev):
        return self.forward(prev)
    
    def forward(self, prev):
        raise NotImplementedError

    def backward(self, dx):
        pass

    def grad_descent(self, alpha):
        pass

In [4]:
class Flatten(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, prev):
        self.x = prev.reshape(prev.shape[0], -1)
        
        return self

In [ ]:
class ReLU(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, prev):
        self.x = np.maximum(prev.x, 0)
        self.prev_layer = prev
        
        return self
    
    def backward(self, dx):
        if not self.prev_layer: return
        
        self.prev_layer.backward(
            dx * (self.prev_layer.x > 0)
        )

    def grad_descent(self, alpha):
        if self.prev_layer:
            self.prev_layer.grad_descent(alpha)

In [ ]:
class Linear(Layer):
    def __init__(self, n_in, n_out):
        super().__init__()

        self.w = np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)
        self.b = np.zeros(n_out)

    def forward(self, prev):
        self.prev_layer = prev
        self.x = self.prev_layer.x @ self.w + self.b

        return self
    
    def backward(self, dx):
        self.db = np.sum(dx, axis=0) / dx.shape[0]

        if not self.prev_layer: return

        self.dw = self.prev_layer.x.T @ dx / dx.shape[0]
        self.prev_layer.backward(dx @ self.w.T)

    def grad_descent(self, alpha):
        self.w -= self.dw*alpha
        self.b -= self.db*alpha

        if self.prev_layer:
            self.prev_layer.grad_descent(alpha)


In [ ]:
from scipy import signal

class Conv2D(Layer):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()

        fan_in = kernel_size*kernel_size * in_channels
        std = np.sqrt(2.0/fan_in)

        self.kernel = np.random.randn(in_channels, out_channels, kernel_size, kernel_size) * std
        self.bias = np.zeros((out_channels, 1, 1))

    def forward(self, prev):
        self.prev_layer = prev if isinstance(prev, Layer) else None
        self.prev_x = prev.x if self.prev_layer is not None else prev

        if len(self.prev_x.shape) == 2:
            self.prev_x = self.prev_x[np.newaxis, :, :]

        in_channels, out_channels, _, _ = self.kernel.shape

        _, prev_x_h, prev_x_w = self.prev_x.shape

        self.x = np.zeros((out_channels, prev_x_h, prev_x_w))

        for out_c in range(out_channels):
            for in_c in range(in_channels):
                curr_kernel = self.kernel[in_c, out_c, :, :]

                self.x[out_c] += signal.correlate2d(
                    self.prev_x[in_c],
                    curr_kernel,
                    mode='same',
                    boundary='fill',
                    fillvalue=0
                )

            self.x[out_c] += self.bias[out_c]

        return self

    def backward(self, dx):
        in_channels, out_channels, k_size, _ = self.kernel.shape

        pad = k_size//2

        self.dkernel = np.zeros_like(self.kernel)
        self.dbias = np.zeros((out_channels, 1, 1))

        for out_c in range(out_channels):
            self.dbias[out_c] = np.sum(dx[out_c])

            for in_c in range(in_channels):
                prev_x_padded = np.pad(
                    self.prev_x[in_c], 
                    pad,
                    mode='constant'
                )

                self.dkernel[in_c, out_c] = signal.correlate2d(
                    prev_x_padded, dx[out_c], mode='valid'
                )

        if not self.prev_layer: return
        
        prev_dx = np.zeros_like(self.prev_layer.x)
        for out_c in range(out_channels):
            for in_c in range(in_channels):
                prev_dx[in_c] += signal.convolve2d(
                    dx[out_c], self.kernel[in_c, out_c], mode='same'
                )

        self.prev_layer.backward(prev_dx)

    def grad_descent(self, alpha):
        self.kernel -= self.dkernel*alpha
        self.bias -= self.dbias*alpha

        if self.prev_layer:
            self.prev_layer.grad_descent(alpha)
    

In [ ]:
class MaxPool2D(Layer):
    def __init__(self, size=2):
        super().__init__()

        self.size=size

    def forward(self, prev):
        self.prev_layer = prev
        prev_x = prev.x

        if len(prev_x.shape) == 2:
            prev_x = prev_x[np.newaxis, :, :]

        size = self.size

        channels, width, height = prev_x.shape
        new_width = -(width // -size)
        new_height = -(height // -size)

        self.x = np.zeros((
            channels,
            new_width,
            new_height
        ))

        self.mask = np.zeros_like(prev_x)

        for c in range(channels):
            for i in range(new_width):
                for j in range(new_height):
                    w_start = i*size
                    w_end = w_start+size

                    h_start = j*size
                    h_end = h_start+size

                    region = prev_x[c, w_start:w_end, h_start:h_end]
                    self.x[c, i, j] = np.max(region)

                    idx_1d = np.argmax(region)
                    local_r, local_c = np.unravel_index(idx_1d, region.shape)

                    global_r, global_c = w_start+local_r, h_start+local_c

                    self.mask[c, global_r, global_c] = 1

        return self

    def backward(self, dx):
        size = self.size
        channels, new_width, new_height = self.x.shape

        prev_dx = np.zeros_like(self.mask)

        for c in range(channels):
            for i in range(new_width):
                for j in range(new_height):
                    w_start = i*size
                    w_end = w_start+size

                    h_start = j*size
                    h_end = h_start+size 

                    prev_dx[c, w_start:w_end, h_start:h_end] = (
                        dx[c, i, j] * self.mask[c, w_start:w_end, h_start:h_end]
                    )
        self.prev_layer.backward(prev_dx)


In [13]:
def softmax(z):
    z = np.asarray(z)
    e_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e_z / np.sum(e_z, axis=1, keepdims=True)

class CrossEntropy():
    def __init__(self, logits_linear, y):
        self.y = y
        self.logits_linear = logits_linear
        self.y_cap = softmax(logits_linear.x)

        self.value = -np.sum(self.y*np.log(self.y_cap + 1e-15))

    def backward(self):
        self.logits_linear.backward(self.y_cap - self.y)

    def grad_descent(self, alpha):
        self.logits_linear.grad_descent(alpha)

In [10]:
def one_hot(y, num_classes=10):
    res = np.zeros((len(y), num_classes))
    res[np.arange(len(y)), y] = 1
    return res

In [17]:
class NN:
    def __init__(self):
        self.flatten = Flatten()
        self.fc1 = Linear(28*28, 64)
        self.relu12 = ReLU()
        self.fc2 = Linear(64, 32)
        self.relu23 = ReLU()
        self.fc3 = Linear(32, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu12(self.fc1(x))
        x = self.relu23(self.fc2(x))
        x = self.fc3(x)

        return x

    def predict(self, Xt):
        Xt = np.asarray(Xt, dtype=np.float64) / 255.0
        logits_linear = self.forward(Xt)
        return np.argmax(logits_linear.x, axis=1)

    def fit(self, Xt, yt, epochs=50, alpha=0.1, batch_size=32, verbose=0):
        Xt = np.asarray(Xt, dtype=np.float64)/255.0
        yt = one_hot(yt)
        
        n = Xt.shape[0]

        rng = np.random.default_rng(42)

        for epoch in range(epochs):
            indices = rng.permutation(n)
            Xt = Xt[indices]
            yt = yt[indices]

            l = 0

            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                X_batch = Xt[start:end]
                y_batch = yt[start:end]

                logits_linear = self.forward(X_batch)
                loss = CrossEntropy(logits_linear, y_batch)
                loss.backward()
                loss.grad_descent(alpha)

                l += loss.value

            if verbose and ((epoch+1)%verbose == 1 or epoch+1==epochs):
                print(f"epoch {epoch+1}/{epochs}. Loss: {l/n}")

        print("done.")

In [18]:
nn = NN()

nn.fit(X_train, y_train, alpha=0.1, epochs=50, batch_size=64, verbose=10)

epoch 1/50. Loss: 0.43148006904158265
epoch 11/50. Loss: 0.045836097967520015
epoch 21/50. Loss: 0.011241670530623409
epoch 31/50. Loss: 0.0027771434166960146
epoch 41/50. Loss: 0.0014147599520005806
epoch 50/50. Loss: 0.0009500777070916352
done.


In [ ]:
def accuracy(y_cap, y):
    return np.mean(y_cap==y)

y_cap = nn.predict(X_val)
print(accuracy(y_cap, y_val))

0.97


In [ ]:
X_test = pd.read_csv("test.csv")
y_test_cap = nn.predict(X_test)

res = pd.DataFrame({
    "ImageId": range(1, len(y_test_cap)+1),
    "Label": y_test_cap
})

res.to_csv("submission.csv", index=False)